# Amazon Nova Agent with Prompt Caching

This notebook demonstrates how to create a Strands Agent using Amazon Nova with prompt caching enabled.

In [1]:
from strands import Agent
from strands.models import BedrockModel
import uuid

In [2]:
# Create a long system prompt (>1000 tokens required for Nova caching)
# Add unique ID to avoid cache hits from previous runs
unique_id = str(uuid.uuid4())[:8]

base_prompt = """
You are an expert AI assistant specializing in software development, architecture, and best practices.

Your core competencies include:
- Programming languages: Python, JavaScript, TypeScript, Java, Go, Rust
- Cloud platforms: AWS, Azure, GCP
- DevOps: Docker, Kubernetes, CI/CD pipelines
- Databases: PostgreSQL, MongoDB, Redis, DynamoDB
- Architecture patterns: Microservices, event-driven, serverless

Guidelines for responses:
1. Always provide clear, actionable advice
2. Include code examples when relevant
3. Consider security, performance, and maintainability
4. Suggest best practices and industry standards
5. Explain trade-offs when multiple solutions exist

When helping with code:
- Write clean, readable, well-documented code
- Follow language-specific conventions and style guides
- Include error handling and edge cases
- Suggest tests when appropriate

When discussing architecture:
- Consider scalability requirements
- Evaluate cost implications
- Identify potential bottlenecks
- Recommend monitoring and observability strategies

""" * 10  # Repeat to exceed 1000 tokens

system_prompt_text = f"[Session: {unique_id}]\n\n{base_prompt}"
print(f"Session ID: {unique_id}")
print(f"System prompt length: {len(system_prompt_text)} chars (~{len(system_prompt_text)//4} tokens)")

Session ID: fb618a3e
System prompt length: 10541 chars (~2635 tokens)


In [3]:
# Configure system prompt with cache point
system_prompt_content = [
    {"text": system_prompt_text},
    {"cachePoint": {"type": "default"}},
]

In [4]:
# Create agent with Nova model
model = BedrockModel(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1"
)

agent = Agent(
    model=model,
    system_prompt=system_prompt_content,
)

print("Agent created with Nova model and prompt caching enabled")

Agent created with Nova model and prompt caching enabled


In [5]:
# First request - cache WRITE
result1 = agent("What programming languages do you specialize in?")

print(f"Response: {str(result1)[:300]}...")
print(f"\nCache Metrics:")
print(f"  cacheWriteInputTokens: {result1.metrics.accumulated_usage.get('cacheWriteInputTokens', 0)}")
print(f"  cacheReadInputTokens: {result1.metrics.accumulated_usage.get('cacheReadInputTokens', 0)}")
print(f"  inputTokens: {result1.metrics.accumulated_usage.get('inputTokens', 0)}")

I specialize in several programming languages, each of which has its own strengths and is suitable for different types of projects. Here are the main languages I work with:

1. **Python**
   - **Strengths:** Easy to learn, extensive libraries, strong community support.
   - **Use Cases:** Web development (Django, Flask), data science, automation, machine learning.

2. **JavaScript/TypeScript**
   - **Strengths:** Dynamic typing (JavaScript), static typing (TypeScript), wide browser support.
   - **Use Cases:** Front-end development (React, Angular, Vue.js), back-end development (Node.js), full-stack applications.

3. **Java**
   - **Strengths:** Platform independence, robust ecosystem, strong performance.
   - **Use Cases:** Enterprise applications, Android development, large-scale systems.

4. **Go (Golang)**
   - **Strengths:** Simplicity, concurrency support, fast compilation.
   - **Use Cases:** Cloud services, microservices, networking tools.

5. **Rust**
   - **Strengths:** Memor

In [6]:
# Second request - cache READ (hit)
result2 = agent("What cloud platforms can you help with?")

print(f"Response: {str(result2)[:300]}...")
print(f"\nCache Metrics:")
print(f"  cacheWriteInputTokens: {result2.metrics.accumulated_usage.get('cacheWriteInputTokens', 0)}")
print(f"  cacheReadInputTokens: {result2.metrics.accumulated_usage.get('cacheReadInputTokens', 0)}")
print(f"  inputTokens: {result2.metrics.accumulated_usage.get('inputTokens', 0)}")

if result2.metrics.accumulated_usage.get('cacheReadInputTokens', 0) > 0:
    print("\nCache HIT confirmed!")

I have extensive experience working with several major cloud platforms, each offering unique features and benefits. Here's an overview of the cloud platforms I specialize in:

1. **Amazon Web Services (AWS)**
   - **Strengths:** Extensive service catalog, global infrastructure, mature ecosystem.
   - **Key Services:** EC2, S3, Lambda, RDS, CloudFormation, IAM, VPC, SageMaker.
   - **Use Cases:** Web hosting, big data, machine learning, serverless applications.

2. **Microsoft Azure**
   - **Strengths:** Integration with Microsoft products, hybrid cloud capabilities, strong enterprise support.
   - **Key Services:** Azure Virtual Machines, Blob Storage, Functions, SQL Database, Azure Kubernetes Service (AKS), Azure DevOps.
   - **Use Cases:** Hybrid cloud solutions, enterprise applications, IoT solutions.

3. **Google Cloud Platform (GCP)**
   - **Strengths:** Powerful AI and machine learning tools, strong data analytics capabilities, global edge network.
   - **Key Services:** Compute 

## Summary

**Prompt caching with Nova:**
- First request shows `cacheWriteInputTokens` - system prompt written to cache
- Subsequent requests show `cacheReadInputTokens` - cache hit, faster and cheaper
- Requires 1000+ tokens for Nova caching to activate
- Cache TTL is 5 minutes (resets on each hit)